# WCF sample-size sensitivity shard 13

Shard 13 of 51 contains 33 cells, methods `causal_drf`, `cwdb_dr`, `cwdb_zipt`, `drf`, source DGPs D2, D5, IC0, IC1, ZI0, ZI1, ZI2, and sample sizes [125, 250, 500, 1000].

The rough reference cost is 57 minutes. The estimate is only for balancing; Colab is usually slower. This notebook embeds its source and manifest slice, checkpoints after each cell, and ends by downloading one zip bundle.


In [ ]:
import os
for _variable in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS',
                  'NUMEXPR_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS', 'R_NUM_THREADS'):
    os.environ[_variable] = '1'
print('numerical libraries pinned to one thread')


## Embedded source

In [ ]:
import base64
import pathlib
import sys
import tempfile
import zipfile

SOURCE_ARCHIVE_SHA256 = 'e7d332f2d80ce87e779a037e47181035d4294a890567966b0f6e5be5a35dd74c'
SOURCE_ARCHIVE_B64 = '''\
...
Dy4EAAAA
'''
workdir = pathlib.Path(tempfile.mkdtemp(prefix='wcf_sample_size_'))
archive_path = workdir / 'wcf_source.zip'
archive_path.write_bytes(base64.b64decode(SOURCE_ARCHIVE_B64))
actual = __import__('hashlib').sha256(archive_path.read_bytes()).hexdigest()
assert actual == SOURCE_ARCHIVE_SHA256, (actual, SOURCE_ARCHIVE_SHA256)
with zipfile.ZipFile(archive_path) as archive:
    archive.extractall(workdir)
sys.path.insert(0, str(workdir / 'src'))
print('embedded source:', (workdir / 'src').resolve())
print('manifest checksum: 9ac5893362f1d59b7fc41a3519bbc761cc0b5854a94b69f642f9eb997f61ac21')


## R forest dependencies

This setup is needed for the Causal-DRF and DRF comparator cells.

In [ ]:
# The R bridge subprocesses inherit this variable from the notebook process, so
# it must be set in Python rather than in the install shell below.
import os
os.environ['WCF_CAUSAL_DRF_R_LIB'] = '/content/Rlib/causal_drf'
print('WCF_CAUSAL_DRF_R_LIB =', os.environ['WCF_CAUSAL_DRF_R_LIB'])


In [ ]:
# This shard contains R forest cells. The setup installs R, the pinned CRAN
# `drf` 1.3.1, and the authors' causal-clean package at the frozen commit into
# a notebook-local library that WCF_CAUSAL_DRF_R_LIB selects. Expect fifteen
# to twenty-five minutes for this cell.
%%bash
set -e
apt-get -qq update > /dev/null 2>&1
apt-get -qq install -y r-base r-base-dev libcurl4-openssl-dev libssl-dev libxml2-dev curl > /dev/null 2>&1
Rscript -e 'options(Ncpus=2); install.packages(c("Rcpp","RcppEigen","jsonlite","remotes","transport","fastDummies","kernlab"), repos="https://cloud.r-project.org", quiet=TRUE)'
# CRAN drf 1.3.1 drives the paper-DRF baseline; the causal-clean library
# below shadows it only for Causal-DRF cells.
Rscript -e 'options(Ncpus=2); if (!requireNamespace("drf", quietly=TRUE)) install.packages("drf", repos="https://cloud.r-project.org", quiet=TRUE); cat("CRAN drf", as.character(packageVersion("drf")), "ready\n")'
mkdir -p /content/Rlib/causal_drf
CAUSAL_SHA="0a1a508444176b5b1553f13e832be93a374b0af2"
if [ ! -d /content/Rlib/causal_drf/drf ]; then
  TARBALL="/tmp/causal_clean_${CAUSAL_SHA:0:12}.tar.gz"
  curl -sL "https://codeload.github.com/herbps10/drf/tar.gz/${CAUSAL_SHA}" -o "$TARBALL"
  EXTRACT="/tmp/causal_clean_src"
  rm -rf "$EXTRACT"; mkdir -p "$EXTRACT"
  tar -xzf "$TARBALL" -C "$EXTRACT"
  PKG_DIR=$(find "$EXTRACT" -maxdepth 3 -type d -path "*r-package/drf" | head -1)
  echo "installing causal-clean drf from $PKG_DIR"
  R CMD INSTALL --library=/content/Rlib/causal_drf "$PKG_DIR" \
    || Rscript -e 'options(Ncpus=2); .libPaths(c("/content/Rlib/causal_drf",.libPaths())); remotes::install_github("herbps10/drf", ref="0a1a508444176b5b1553f13e832be93a374b0af2", subdir="r-package/drf", lib="/content/Rlib/causal_drf", upgrade="never", quiet=TRUE)'
fi
Rscript -e '.libPaths(c("/content/Rlib/causal_drf",.libPaths())); stopifnot(requireNamespace("drf", quietly=TRUE)); cat("causal-clean drf", as.character(packageVersion("drf")), "ready\n")'
echo 'setup complete'


## Manifest registration

In [ ]:
import json
from wasserstein_causal_forests.g3 import runner
from wasserstein_causal_forests.g3.wcf_sensitivity import apply_method_registry

SHARD_INDEX = 13
SHARD_TOTAL = 51
ESTIMATED_SECONDS = 3449.9350000000004
MANIFEST_SLICE = json.loads('''{"manifest_contract_id": "WCF-SAMPLE-SIZE-ALL-v1", "manifest_checksum": "9ac5893362f1d59b7fc41a3519bbc761cc0b5854a94b69f642f9eb997f61ac21", "estimator_source_hash": "c105c0aaff1b0b8e598799351cfb490845ef2253adffd909364c0670e0505d92", "method_registry": {"cwdb_dr": {"role": "variant", "adapter": "cwdb_dr", "produces_law": true, "cross_fitted": true, "parameters": {"contrast_candidates": [0.0, 50.0, 500.0], "n_folds": 3, "common_grid_levels": "COMMON199+INTERIOR", "propensity_factory": "logistic"}, "propensity_clip": [0.02, 0.98], "boosting_budget": {"n_estimators": 100, "learning_rate": 0.12, "max_depth": 4, "min_samples_leaf": 10, "min_arm_leaf": 5, "collision_epsilon": 0.001}, "estimator_source_hash": "c105c0aaff1b0b8e598799351cfb490845ef2253adffd909364c0670e0505d92"}, "causal_drf": {"role": "baseline", "adapter": "forest", "produces_law": true, "parameters": {"method": "causal_drf"}}, "drf": {"role": "baseline", "adapter": "forest", "produces_law": true, "parameters": {"method": "drf"}}, "cwdb_zipt": {"role": "variant", "adapter": "zipt", "produces_law": true, "target_ids": ["LAW-A-M-K", "LAW-A-K", "MEANQ-A-K", "TATE-K-grid_mean", "TCATE-K-grid_mean", "REF-ATE-K", "REF-TCATE-K"], "inference": null, "cross_fitted": true, "parameters": {"classifier_c": 1.0, "contrast_candidates": [0.0, 50.0, 500.0], "n_folds": 2}}}, "cells": [{"grid": "wcf_sample_size_v1_logit_f3", "dgp": "D2", "n_train": 125, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 1, "cell_key": "f0f58c79727b92f0", "test_seed": 900001}, {"grid": "wcf_sample_size_v1_logit_f3", "dgp": "D2", "n_train": 125, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 1, "cell_key": "91b51f6e4782f337", "test_seed": 900001}, {"grid": "wcf_sample_size_v1_logit_f3", "dgp": "D2", "n_train": 125, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 1, "cell_key": "e648f92b9f5aef7d", "test_seed": 900001}, {"grid": "wcf_sample_size_v1_logit_f3", "dgp": "D2", "n_train": 250, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 3, "cell_key": "791e6ea830683fd2", "test_seed": 900003}, {"grid": "wcf_sample_size_v1_logit_f3", "dgp": "D2", "n_train": 250, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 3, "cell_key": "beaeb1bf54102532", "test_seed": 900003}, {"grid": "wcf_sample_size_v1_logit_f3", "dgp": "D2", "n_train": 250, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 3, "cell_key": "51b2c190a0dec3bb", "test_seed": 900003}, {"grid": "wcf_sample_size_v1_logit_f3", "dgp": "D5", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 8, "cell_key": "cfb13c05f5397699", "test_seed": 900008}, {"grid": "wcf_sample_size_v1_logit_f3", "dgp": "D5", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 8, "cell_key": "11f5f7bf33a89b19", "test_seed": 900008}, {"grid": "wcf_sample_size_v1_logit_f3", "dgp": "D5", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 8, "cell_key": "48e501e7f6518491", "test_seed": 900008}, {"grid": "wcf_sample_size_v1_logit_f3", "dgp": "D5", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 4, "cell_key": "140d79317cce9fdb", "test_seed": 900004}, {"grid": "wcf_sample_size_v1_logit_f3", "dgp": "D5", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 4, "cell_key": "da2e584730339ff3", "test_seed": 900004}, {"grid": "wcf_sample_size_v1_logit_f3", "dgp": "D5", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 4, "cell_key": "66150a1f07cbd66e", "test_seed": 900004}, {"grid": "wcf_sample_size_v1_logit_f3", "dgp": "IC0", "n_train": 125, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 4, "cell_key": "af66d8702a2b077c", "test_seed": 900004}, {"grid": "wcf_sample_size_v1_logit_f3", "dgp": "IC0", "n_train": 125, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 4, "cell_key": "4fde9cab9df8f32f", "test_seed": 900004}, {"grid": "wcf_sample_size_v1_logit_f3", "dgp": "IC0", "n_train": 125, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 4, "cell_key": "27976e2dc31d8ba9", "test_seed": 900004}, {"grid": "wcf_sample_size_v1_logit_f3", "dgp": "IC1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 7, "cell_key": "a68daca035204992", "test_seed": 900007}, {"grid": "wcf_sample_size_v1_logit_f3", "dgp": "IC1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 7, "cell_key": "79f80c3b775b3f08", "test_seed": 900007}, {"grid": "wcf_sample_size_v1_logit_f3", "dgp": "IC1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 7, "cell_key": "c17467626fa5a700", "test_seed": 900007}, {"grid": "wcf_sample_size_v1_logit_f3", "dgp": "IC1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_dr", "seed": 3, "cell_key": "e9b7072f81fe205f", "test_seed": 900003}, {"grid": "wcf_sample_size_v1_logit_f3", "dgp": "IC1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 3, "cell_key": "3006002dd937a43e", "test_seed": 900003}, {"grid": "wcf_sample_size_v1_logit_f3", "dgp": "IC1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 3, "cell_key": "b209ec3c5acb51af", "test_seed": 900003}, {"grid": "wcf_sample_size_v1_logit_f3", "dgp": "ZI0", "n_train": 250, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 2, "cell_key": "6424332d33190f09", "test_seed": 900002}, {"grid": "wcf_sample_size_v1_logit_f3", "dgp": "ZI0", "n_train": 250, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 2, "cell_key": "ba54432099d27389", "test_seed": 900002}, {"grid": "wcf_sample_size_v1_logit_f3", "dgp": "ZI0", "n_train": 250, "n_grid": 25, "n_particles": 10, "method": "cwdb_zipt", "seed": 2, "cell_key": "8d095fa5eef7c1f3", "test_seed": 900002}, {"grid": "wcf_sample_size_v1_logit_f3", "dgp": "ZI0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 4, "cell_key": "fc1aff1df701d972", "test_seed": 900004}, {"grid": "wcf_sample_size_v1_logit_f3", "dgp": "ZI0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 4, "cell_key": "6b2f310af6b191f2", "test_seed": 900004}, {"grid": "wcf_sample_size_v1_logit_f3", "dgp": "ZI0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "cwdb_zipt", "seed": 4, "cell_key": "12776c6cf213fdf5", "test_seed": 900004}, {"grid": "wcf_sample_size_v1_logit_f3", "dgp": "ZI1", "n_train": 125, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 1, "cell_key": "26558d104257e81f", "test_seed": 900001}, {"grid": "wcf_sample_size_v1_logit_f3", "dgp": "ZI1", "n_train": 125, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 1, "cell_key": "dd92da0e5cea984b", "test_seed": 900001}, {"grid": "wcf_sample_size_v1_logit_f3", "dgp": "ZI1", "n_train": 125, "n_grid": 25, "n_particles": 10, "method": "cwdb_zipt", "seed": 1, "cell_key": "6ffb16a248c98c45", "test_seed": 900001}, {"grid": "wcf_sample_size_v1_logit_f3", "dgp": "ZI2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 5, "cell_key": "e02a70b8c1b63886", "test_seed": 900005}, {"grid": "wcf_sample_size_v1_logit_f3", "dgp": "ZI2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 5, "cell_key": "2965082d97595f46", "test_seed": 900005}, {"grid": "wcf_sample_size_v1_logit_f3", "dgp": "ZI2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "cwdb_zipt", "seed": 5, "cell_key": "dc9040c0468a09b5", "test_seed": 900005}]}''')
apply_method_registry({'method_registry': MANIFEST_SLICE['method_registry']})
print('registered methods:', sorted(MANIFEST_SLICE['method_registry']))
print('cells:', len(MANIFEST_SLICE['cells']))


## Run

Re-run this cell after interruption to resume from the checkpoint.

In [ ]:

import json
import os
import time
from pathlib import Path

import pyarrow.parquet as pq
from wasserstein_causal_forests.g3 import runner
from wasserstein_causal_forests.g3.manifest import Cell

OUTPUT = Path('shard_output')
OUTPUT.mkdir(parents=True, exist_ok=True)
PARQUET = OUTPUT / 'wcf_sample_size_parquet.parquet'
LOG = OUTPUT / 'execution_log.jsonl'
FAILURES = OUTPUT / 'failure_rows.jsonl'
CACHE = OUTPUT / 'cache'
CACHE.mkdir(parents=True, exist_ok=True)

def read_rows(path):
    return pq.read_table(path).to_pylist() if path.exists() else []

rows = read_rows(PARQUET)
by_cell = {}
for row in rows:
    by_cell.setdefault(row['cell_key'], []).append(row)
successful = {key for key, group in by_cell.items()
              if not any(row.get('metric') == 'cell_failure' for row in group)}
failed = {key for key, group in by_cell.items()
          if any(row.get('metric') == 'cell_failure' for row in group)}
if failed:
    rows = [row for row in rows if row['cell_key'] not in failed]
    print('retrying failed cells:', len(failed))

started = time.time()
n_ok = n_failed = n_skipped = 0
total_cells = len(MANIFEST_SLICE['cells'])
for position, item in enumerate(MANIFEST_SLICE['cells'], start=1):
    if item['cell_key'] in successful:
        n_skipped += 1
        continue
    cell = Cell(**{key: value for key, value in item.items()
                   if key not in ('cell_key', 'test_seed')})
    assert cell.key == item['cell_key']
    cell_rows = runner.run_cell(cell, cache_directory=CACHE,
                                manifest_contract_id=MANIFEST_SLICE['manifest_contract_id'])
    rows.extend(cell_rows)
    status = 'failed' if cell_rows[0].get('status') == 'failed' else 'ok'
    n_ok += status == 'ok'
    n_failed += status == 'failed'
    temporary = PARQUET.with_suffix('.tmp')
    runner.write_rows(rows, temporary)
    os.replace(temporary, PARQUET)
    with LOG.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps({'cell_key': cell.key, 'status': status,
                                 'n_rows': len(cell_rows),
                                 'wall_seconds': float(cell_rows[0].get('wall_seconds', 0.0)),
                                 'finished_at': time.time()}) + '\n')
    if status == 'failed':
        with FAILURES.open('a', encoding='utf-8') as handle:
            handle.write(json.dumps({'cell_key': cell.key, 'rows': cell_rows}, default=str) + '\n')
    print(f'[{position}/{total_cells}] {cell.dgp} n={cell.n_train} '
          f'{cell.method} seed={cell.seed}: {status}', flush=True)
print(f'finished: {n_ok} ok, {n_failed} failed, {n_skipped} skipped; '
      f'{(time.time() - started) / 60.0:.1f} minutes')


## Diagnostics and sidecar

In [ ]:
import json
import platform
import sys
import time
from pathlib import Path

versions = {'python': sys.version.split()[0], 'platform': platform.platform()}
for _name in ('numpy', 'scipy', 'sklearn', 'pandas', 'pyarrow'):
    try:
        _module = __import__(_name)
        versions[_name] = getattr(_module, '__version__', 'unknown')
    except ImportError:
        versions[_name] = None
config = {'shard_index': 13, 'shard_total': 51, 'manifest_contract_id': MANIFEST_SLICE['manifest_contract_id'], 'manifest_checksum': MANIFEST_SLICE['manifest_checksum'], 'source_archive_sha256': 'e7d332f2d80ce87e779a037e47181035d4294a890567966b0f6e5be5a35dd74c', 'n_cells': len(MANIFEST_SLICE['cells']), 'versions': versions, 'estimated_seconds_reference': ESTIMATED_SECONDS}
out = Path('shard_output')
(out / 'shard_config.json').write_text(json.dumps(config, indent=2), encoding='utf-8')
(out / 'manifest_slice.json').write_text(json.dumps(MANIFEST_SLICE, indent=2), encoding='utf-8')
(out / 'wcf_sample_size_parquet.meta.json').write_text(json.dumps({'manifest_checksum': MANIFEST_SLICE['manifest_checksum'], 'estimator_source_hash': MANIFEST_SLICE['estimator_source_hash'], 'contract_id': MANIFEST_SLICE['manifest_contract_id'], 'updated_at': time.time()}, indent=2), encoding='utf-8')
print(json.dumps(config, indent=2))


## Download

In [ ]:
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

output_file = 'wcf_sample_size_shard.zip'
with ZipFile(output_file, 'w', ZIP_DEFLATED) as archive:
    for path in Path('shard_output').rglob('*'):
        if path.is_file():
            archive.write(path, arcname=path.relative_to('shard_output'))
try:
    from google.colab import files
    files.download(output_file)
    print('Downloaded:', output_file)
except Exception as e:
    print('(Not on Colab / download skipped):', e)
